In [ ]:
import numpy as np

def genMat(seedA, n, q):
    np.random.seed(seedA)
    A = np.random.randint(0, q, size=(n, n), dtype=np.int64)
    return A

In [ ]:
T_CHI640 = [4643, 13363, 20579, 25843, 29227, 31145, 32103, 32525, 32689, 32745, 32762, 32767]

In [ ]:
def chiSample(T_CHI, n1, n2, k):
    matrix = np.zeros((n1, n2), dtype=np.int64)
    upper_bound = 2**k 
    
    for r in range(n1):
        for c in range(n2):
            x = np.random.randint(0, upper_bound)
            
            sample_value = 0
            for i, cumulative_prob in enumerate(T_CHI):
                if x < cumulative_prob:
                    sample_value = i
                    break
            else:
                sample_value = len(T_CHI)
            
            sinal = np.random.randint(0, 2)
            if sinal == 1 and sample_value > 0:
                sample_value = -sample_value
                
            matrix[r, c] = sample_value
            
    return matrix

In [ ]:
import numpy as np

def encode(msgBits, B, q):
    M = np.zeros((8, 8), dtype=np.int64)
    scale = q // (2**B)
    
    bit_i = 0
    for i in range(8):
        for j in range(8):
            chunk = msgBits[bit_i : bit_i + B]
            
            x = 0
            for k in range(len(chunk)):
                x += chunk[k] * (2**k)
            
            M[i, j] = x * scale
            bit_i += B
            
    return M

In [ ]:
def decode(M, B, q):
    msgBits = []
    n_bar, m_bar = M.shape 

    for i in range(n_bar):
        for j in range(m_bar):
            x = M[i, j]

            # v = round( (x * 2^B) / q )
            v = int(round((x * (2**B)) / q)) % (2**B)

            for k in range(B):
                bit = (v >> k) & 1
                msgBits.append(bit)

    return msgBits

In [ ]:
def miniPKE_KeyGen():
    # n = 64 (dimensão grande), q = 1024
    seedA = np.random.randint(0, 1024)
    
    A = genMat(seedA, 64, 1024) 
    
    # O segredo e o erro são n x n_bar (64x8)
    T_CHI64 = [6536, 14465, 16234, 16379, 16384]
    segredo = chiSample(T_CHI64, 64, 8, k=14)
    erro = chiSample(T_CHI64, 64, 8, k=14)
    
    B_pk = (np.matmul(A, segredo) + erro) % 1024
    
    return segredo, (seedA, B_pk)

In [ ]:
def miniPKE_Enc(pk, msgBits):
    seedA, B_pk = pk
    A = genMat(seedA, n=64, q=1024)
    T_CHI64 = [6536, 14465, 16234, 16379, 16384]
    
    # ruídos de quem envia
    sLinha = chiSample(T_CHI64, 8, 64, k=14)
    eLinha = chiSample(T_CHI64, 8, 64, k=14)
    eDoubleLinha = chiSample(T_CHI64, 8, 8, k=14)
    
    # C1 = S' * A + E'
    C1 = (np.matmul(sLinha, A) + eLinha) % 1024

    # V' = S' * B_pk + E''
    vLinha = (np.matmul(sLinha, B_pk) + eDoubleLinha) % 1024
    
    # Encoding da message e somar a V'
    M = encode(msgBits, B=2, q=1024)
    C2 = (vLinha + M) % 1024
    
    return (C1, C2)

In [ ]:
def miniPKE_Dec(sk, C):
    C1, C2 = C
    S = sk # Private Key
    
    # V = C1 * S
    V = np.matmul(C1, S) % 1024
    
    # Retirar a mask da mensagem
    M = (C2 - V) % 1024
    
    # Decoding para bits
    m_bits = decode(M, B=2, q=1024)
    
    return m_bits

In [ ]:
# Gera as keys
sk, pk = miniPKE_KeyGen()

# Mensagem aleatória de 128 bits
msg = np.random.randint(0, 2, 128).tolist()

criptograma = miniPKE_Enc(pk, msg)
msgRecuperada = miniPKE_Dec(sk, criptograma)

if msg == msgRecuperada:
    print("Funcionou")
else:
    print("Erro no decoding")

In [ ]:
def miniPKE_KeyGen_Ruido():
    seedA = np.random.randint(0, 1024)
    A = genMat(seedA, 64, 1024) 

    segredo = chiSample(T_CHI640, 64, 8, k=15)
    erro = chiSample(T_CHI640, 64, 8, k=15)

    B_pk = (np.matmul(A, segredo) + erro) % 1024
    return segredo, (seedA, B_pk)

In [ ]:
def miniPKE_Enc_Ruido(pk, msgBits):
    seedA, B_pk = pk
    A = genMat(seedA, n=64, q=1024)
    
    # ruido mais intenso para um sistema de 64
    sLinha = chiSample(T_CHI640, 8, 64, k=15)
    eLinha = chiSample(T_CHI640, 8, 64, k=15)
    eDoubleLinha = chiSample(T_CHI640, 8, 8, k=15)
    
    C1 = (np.matmul(sLinha, A) + eLinha) % 1024
    vLinha = (np.matmul(sLinha, B_pk) + eDoubleLinha) % 1024
    
    M = encode(msgBits, B=2, q=1024)
    C2 = (vLinha + M) % 1024
    
    return (C1, C2)

In [ ]:
# Teste com o ruído forte
sk_n, pk_n = miniPKE_KeyGen_Ruido()
msg_n = np.random.randint(0, 2, 128).tolist()

cripto_n = miniPKE_Enc_Ruido(pk_n, msg_n)
msgRec_n = miniPKE_Dec(sk_n, cripto_n) # Usamos o mesmo Dec anterior

if msg_n == msgRec_n:
    print("Funcionou (inesperadamente)")
else:
    # Conta quantos bits estão errados
    erros = sum(1 for i, j in zip(msg_n, msgRec_n) if i != j)
    print(f"Erro no decoding: {erros} bits errados em 128.")

In [ ]:
def Ruido_Vs_Silencio(iteracoes=100):
    erros_normal = 0
    erros_ruidoso = 0
    
    print(f"A iniciar teste comparativo ({iteracoes} iterações)...")
    
    for _ in range(iteracoes):
        msg = np.random.randint(0, 2, 128).tolist()
        
        # miniPKE Normal (T_CHI64, k=14)
        sk, pk = miniPKE_KeyGen()
        c = miniPKE_Enc(pk, msg)
        msg_rec = miniPKE_Dec(sk, c)
        erros_normal += sum(1 for i, j in zip(msg, msg_rec) if i != j)
        
        # miniPKE Ruidoso (T_CHI640, k=15)
        sk_n, pk_n = miniPKE_KeyGen_Ruido()
        c_n = miniPKE_Enc_Ruido(pk_n, msg)
        msg_rec_n = miniPKE_Dec(sk_n, c_n)
        erros_ruidoso += sum(1 for i, j in zip(msg, msg_rec_n) if i != j)

    # Cálculo das médias
    media_normal = erros_normal / iteracoes
    media_ruidoso = erros_ruidoso / iteracoes
    
    print("\n" + "="*40)
    print(f"{'PARÂMETRO':<20} | {'NORMAL':<10} | {'RUIDOSO':<10}")
    print("-" * 40)
    print(f"{'Média de Bits Errados':<20} | {media_normal:<10.2f} | {media_ruidoso:<10.2f}")
    print(f"{'Taxa de Sucesso (%)':<20} | {((128-media_normal)/128*100):<10.1f}% | {((128-media_ruidoso)/128*100):<10.1f}%")
    print("="*40)

# Executar o teste
Ruido_Vs_Silencio(100)

##### Comentários dos impactos na correcção do algoritmo
Ao substituir o T_CHI64 pelo T_CHI640, o algoritmo miniPKE deixou de ser funcional, apresentando uma taxa de erro de decifração significativa (por volta de 15% dos bits da mensagem).